# Ensemble control-calibration (Q2): global normalization + background subtraction

`region_contrasts.score_regions` compares a numerator condition against a denominator
condition per region. Two control-calibration options remove confounders **before** the
comparison:

1. **Global normalization** (`normalization_mode="global"`) subtracts each sample's
   genome-wide methylation-level offset, so a sample that is globally hotter/colder does
   not masquerade as region-level difference.
2. **Background subtraction** (`ContrastSpec(mode="background_adjusted", ...)`) uses a
   negative-control condition to correct each region's signal as
   `corrected = clamp((p - b) / (1 - b), 0, 1)` (specificity / background-corrected
   occupancy), then runs the same replicate-level beta-binomial LRT + BH.

Both operate on the ensemble pileup counts. This notebook demonstrates the API on the
CTCF demo BAMs; the `background_adjusted` section needs a dedicated negative-control
sample, so treat it as a template you point at your own control.


In [ ]:
from pathlib import Path

from dimelo import global_analysis, parse_bam, plotting, plotting_matplotlib, region_contrasts
from dimelo.models import ContrastSpec, SampleSpec

DATA_DIR = Path("dimelo/test/data")
OUT_DIR = Path("artifacts/region_contrast_control_calibration")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ref_genome = Path("dimelo/test/output/chm13.draft_v1.0.fasta")
analysis_region = "chr16:63442391-63446452"
region_bed = OUT_DIR / "region.bed"
chrom, span = analysis_region.split(":")
start, end = span.split("-")
region_bed.write_text(f"{chrom}\t{start}\t{end}\n")

# barcode17 = targeted experiment, barcode18 = control (see dmr_multi_sample.ipynb).
bams = {
    "barcode17": DATA_DIR / "barcode17.merged.sorted.ctcf_demo.sorted.bam",
    "barcode18": DATA_DIR / "barcode18.merged.sorted.ctcf_demo.sorted.bam",
}


## Build pileups and `SampleSpec`s

`score_regions` reads per-sample bedMethyl pileups via `metadata["pileup_path"]`. Each
`SampleSpec` carries a `condition` (used to define the contrast sides) and a `replicate`.


In [ ]:
pileups = {}
for sample_id, bam in bams.items():
    pileup_path, _ = parse_bam.pileup(
        bam,
        output_name=f"{sample_id}_A",
        output_directory=OUT_DIR,
        ref_genome=ref_genome,
        regions=[analysis_region],
        motifs=["A,0"],
        thresh=0.5,
        cores=2,
        log=False,
        cleanup=True,
        overwrite=False,
        quiet=True,
        override_checks=True,
    )
    pileups[sample_id] = pileup_path

# extract_h5 is a required SampleSpec field but is unused on the ensemble pileup path
# (score_regions reads metadata["pileup_path"]); pass a placeholder.
samples = [
    SampleSpec(sample_id="barcode17", condition="targeted", replicate=1,
               extract_h5="unused.h5",
               metadata={"pileup_path": str(pileups["barcode17"])}),
    SampleSpec(sample_id="barcode18", condition="control", replicate=1,
               extract_h5="unused.h5",
               metadata={"pileup_path": str(pileups["barcode18"])}),
]


## 1. Global normalization

Compute per-(sample, motif) global factors, then pass them to `score_regions` with
`normalization_mode="global"`. The offset (`global_fraction - coverage-weighted pooled
reference_fraction`) is subtracted from each sample's per-region fraction before the
contrast, and the applied mode is recorded in `result.metadata["normalization_mode"]`.


In [ ]:
global_summary = global_analysis.summarize_global_samples(samples=samples, motifs=["A,0"])
factors = global_analysis.compute_global_normalization_factors(global_summary)

contrast = ContrastSpec(mode="pairwise", numerator=["targeted"], denominator=["control"])

result_raw = region_contrasts.score_regions(
    samples=samples, regions=str(region_bed), motifs=["A,0"], contrast=contrast,
    test="beta_binomial",
)
result_norm = region_contrasts.score_regions(
    samples=samples, regions=str(region_bed), motifs=["A,0"], contrast=contrast,
    test="beta_binomial",
    normalization_mode="global", global_normalization_factors=factors,
)

print("raw normalization_mode :", result_raw.metadata["normalization_mode"])
print("norm normalization_mode:", result_norm.metadata["normalization_mode"])
display(result_norm.summary[["region_id", "fraction", "reference_fraction",
                             "delta_fraction", "log2_fc", "p_value", "adjusted_p_value"]])


## 2. Background subtraction (`background_adjusted`)

`background_adjusted` needs a **negative-control** condition in addition to
numerator/denominator (enforced by `ContrastSpec`). Point `background=[...]` at a
no-target control sample. The corrected result carries `background_fraction` /
`background_present`, and the pre-correction fractions are exposed under
`plot_data["region_effect_sizes_raw"]` for the raw-vs-corrected overlay.

Template (swap in your own control sample as a third `SampleSpec` with a distinct
`condition`, e.g. `"igg"`):


In [ ]:
# samples_bg = samples + [
#     SampleSpec(sample_id="igg_control", condition="igg", replicate=1,
#                extract_h5="unused.h5",
#                metadata={"pileup_path": "<path to negative-control pileup>"}),
# ]
# bg_contrast = ContrastSpec(
#     mode="background_adjusted",
#     numerator=["targeted"], denominator=["control"], background=["igg"],
# )
# result_bg = region_contrasts.score_regions(
#     samples=samples_bg, regions=str(region_bed), motifs=["A,0"],
#     contrast=bg_contrast, test="beta_binomial",
#     normalization_mode="global", global_normalization_factors=factors,  # optional
# )
# display(result_bg.summary[["region_id", "fraction", "reference_fraction",
#                            "background_fraction", "background_present",
#                            "delta_fraction", "adjusted_p_value"]])


### Raw-vs-corrected overlay

`prepare_region_contrast_correction_overlay_data` + the matplotlib renderer show, per
region, the effect size before and after background subtraction.


In [ ]:
# overlay = plotting.prepare_region_contrast_correction_overlay_data(result=result_bg)
# display(overlay["overlay_table"])
# fig, _ = plotting_matplotlib.plot_region_contrast_correction_overlay_matplotlib(
#     overlay, title="Background correction: raw vs corrected"
# )
# fig  # noqa: B018


## 3. Single-molecule true-signal calling (Q3)

`dimelo.background` calls each **target read** as true signal or background against a
beta-binomial null fit to **negative-control** reads. Per site, a per-read
`(modified_count, valid_count)` evidence table (from
`region_contrasts.build_single_read_mod_fraction_evidence_table`, whose input is a
per-read extract) drives:

- `call_true_signal_reads` -> per-read `p_value`, BH `q_value`, `is_true_signal`, and a
  soft `occupancy_posterior`;
- `summarize_site_occupancy` -> per-site control-calibrated occupancy rate (feeds Q6);
- `background_removed_pileup` -> the background-removed signal track.

The cell below runs on a small inline evidence table so it executes without a BAM; for
real data, build `evidence` from your extract with
`build_single_read_mod_fraction_evidence_table` (columns `region_id, sample_id,
condition, read_id, modified_count, valid_count`).


In [ ]:
import pandas as pd
from dimelo import background

# Inline per-read evidence: control reads are low-methylation; some target reads carry
# clear single-molecule signal. Replace with build_single_read_mod_fraction_evidence_table
# output for real data.
rows = []
for i, k in enumerate([0, 1, 0, 1, 2, 0, 1, 0]):
    rows.append(dict(region_id="ctcf_site", condition="control", read_id=f"c{i}",
                     modified_count=k, valid_count=20))
for i, k in enumerate([17, 1, 18, 2, 16, 0]):
    rows.append(dict(region_id="ctcf_site", condition="targeted", read_id=f"t{i}",
                     modified_count=k, valid_count=20))
evidence_sm = pd.DataFrame(rows)

called = background.call_true_signal_reads(
    evidence=evidence_sm, target_conditions=["targeted"], control_conditions=["control"],
    fdr=0.05, min_control_reads=5,
)
display(called[["read_id", "read_mod_fraction", "background_rate",
                "p_value", "q_value", "is_true_signal", "occupancy_posterior"]])

occupancy = background.summarize_site_occupancy(called)
display(occupancy)
display(background.background_removed_pileup(called, weighting="hard"))


In [ ]:
from dimelo import plotting, plotting_matplotlib

raster = plotting.prepare_true_signal_read_data(called_reads=called,
                                                color_by="occupancy_posterior")
plotting_matplotlib.plot_true_signal_read_raster_matplotlib(
    raster, title="Single-molecule reads at ctcf_site (posterior)")

track = plotting.prepare_occupancy_rate_track_data(site_occupancy=occupancy)
plotting_matplotlib.plot_occupancy_rate_track_matplotlib(track)

overlay = plotting.prepare_background_removed_pileup_overlay_data(called_reads=called)
plotting_matplotlib.plot_background_removed_pileup_overlay_matplotlib(overlay)
